# US 3.2 -- Baseline CycleGAN Training

This notebook walks through the CycleGAN training loop step by step.  We cover
all the moving parts that make CycleGAN training work: the adversarial losses,
cycle consistency, identity loss, the image pool trick, and learning rate
scheduling.

## What you will learn

1. The three loss components and their weights
2. The alternating generator/discriminator training loop
3. The image pool trick for stable discriminator training
4. Linear LR decay scheduling
5. Running training via the `udm-epic3 train` CLI

## Prerequisites

- Python 3.12, PyTorch
- Install: `uv pip install -e ".[epic3]"`
- Data prepared (see `epic3_01_data_prep.ipynb`)

---
## 1. Loss Components

CycleGAN training uses three loss terms:

### 1.1 Adversarial Loss (LSGAN)

Instead of the original GAN cross-entropy loss, CycleGAN uses least-squares GAN
(LSGAN) for more stable training:

$$L_{adv}(G, D) = \mathbb{E}[(D(G(x)) - 1)^2]$$

The discriminator tries to output 1 for real images and 0 for fakes.
The generator tries to make the discriminator output 1 for its fakes.

### 1.2 Cycle Consistency Loss

The key insight of CycleGAN: if we translate A -> B -> A, we should get back
the original image:

$$L_{cyc}(G_{AB}, G_{BA}) = \mathbb{E}[\|G_{BA}(G_{AB}(a)) - a\|_1] + \mathbb{E}[\|G_{AB}(G_{BA}(b)) - b\|_1]$$

This prevents mode collapse and ensures the generators preserve content.

### 1.3 Identity Loss

If we feed a USM image to the AOI->USM generator, it should not change it:

$$L_{idt}(G_{AB}) = \mathbb{E}[\|G_{AB}(b) - b\|_1]$$

This helps preserve color and tone, especially useful for modalities that share
some visual characteristics.

### Total loss

$$L = L_{adv} + \lambda_{cyc} \cdot L_{cyc} + \lambda_{idt} \cdot L_{idt}$$

Default weights: `lambda_cyc=10.0`, `lambda_idt=0.5`.

In [ ]:
import torch
import torch.nn as nn

from udm_epic3.models.generator import ResnetGenerator
from udm_epic3.models.discriminator import PatchDiscriminator
from udm_epic3.models.cyclegan import CycleGANModel
from udm_epic3.models.losses import (
    adversarial_loss_lsgan,
    cycle_consistency_loss,
    identity_loss,
)
from udm_epic3.data.image_pool import ImagePool

---
## 2. Loss Functions in Action

Let's compute each loss on random data to understand the shapes and magnitudes.

In [ ]:
# Create a small model for demo purposes
G_AB = ResnetGenerator(in_channels=3, out_channels=3, n_residual_blocks=3)
G_BA = ResnetGenerator(in_channels=3, out_channels=3, n_residual_blocks=3)
D_A = PatchDiscriminator(in_channels=3, n_layers=3)
D_B = PatchDiscriminator(in_channels=3, n_layers=3)

# Random input images
real_A = torch.randn(2, 3, 128, 128)  # AOI
real_B = torch.randn(2, 3, 128, 128)  # USM

# Generate fakes
fake_B = G_AB(real_A)   # AOI -> fake USM
fake_A = G_BA(real_B)   # USM -> fake AOI
rec_A = G_BA(fake_B)    # fake USM -> reconstructed AOI
rec_B = G_AB(fake_A)    # fake AOI -> reconstructed USM

# 1. Adversarial loss (generator wants discriminator to output 1.0)
pred_fake_B = D_B(fake_B)
loss_adv = adversarial_loss_lsgan(pred_fake_B, target_is_real=True)
print(f"Adversarial loss (G_AB):   {loss_adv.item():.4f}")

# 2. Cycle consistency loss
loss_cyc = cycle_consistency_loss(rec_A, real_A, rec_B, real_B)
print(f"Cycle consistency loss:    {loss_cyc.item():.4f}")

# 3. Identity loss
idt_B = G_AB(real_B)  # generator should not change real USM
idt_A = G_BA(real_A)  # generator should not change real AOI
loss_idt = identity_loss(idt_A, real_A, idt_B, real_B)
print(f"Identity loss:             {loss_idt.item():.4f}")

---
## 3. Image Pool Trick

Training GANs is notoriously unstable.  One trick from Shrivastava et al. (2017)
is to maintain a **pool of previously generated images** and occasionally show
these older fakes to the discriminator instead of the latest fakes.

This prevents the discriminator from overfitting to the most recent generator
outputs and provides a more stable training signal.

The pool stores up to `pool_size` images (default 50).  When queried:
- 50% chance: return the new image (and add it to the pool)
- 50% chance: return a random old image from the pool (and replace it with the new one)

In [ ]:
# Create an image pool with capacity 50
pool = ImagePool(pool_size=50)

# Simulate filling the pool with generated images
for i in range(60):
    fake_img = torch.randn(1, 3, 128, 128) * i  # different each time
    queried = pool.query(fake_img)

print(f"Pool size: {len(pool.images)} images")
print(f"Pool capacity: {pool.pool_size}")
print(f"\nThe pool returns a mix of new and old generated images,")
print(f"which stabilises discriminator training.")

---
## 4. Training Loop Walkthrough

The CycleGAN training loop alternates between generator and discriminator updates.
Here is a simplified version of one training step:

```python
# ---- Generator update ----
# 1. Generate fakes and reconstructions
fake_B = G_AB(real_A)
fake_A = G_BA(real_B)
rec_A  = G_BA(fake_B)
rec_B  = G_AB(fake_A)

# 2. Compute generator losses
loss_G = (adv_loss(D_B(fake_B), real=True) +
          adv_loss(D_A(fake_A), real=True) +
          lambda_cyc * cycle_loss(rec_A, real_A, rec_B, real_B) +
          lambda_idt * idt_loss(G_AB(real_B), real_B, G_BA(real_A), real_A))

# 3. Update generators
optimizer_G.zero_grad()
loss_G.backward()
optimizer_G.step()

# ---- Discriminator update ----
# 4. Query image pool for historical fakes
fake_B_pool = pool_B.query(fake_B.detach())
fake_A_pool = pool_A.query(fake_A.detach())

# 5. Compute discriminator losses
loss_D_A = 0.5 * (adv_loss(D_A(real_A), real=True) +
                   adv_loss(D_A(fake_A_pool), real=False))
loss_D_B = 0.5 * (adv_loss(D_B(real_B), real=True) +
                   adv_loss(D_B(fake_B_pool), real=False))

# 6. Update discriminators
optimizer_D.zero_grad()
(loss_D_A + loss_D_B).backward()
optimizer_D.step()
```

Key points:
- Generators and discriminators have **separate** optimizers
- Discriminators train on **detached** fakes (no gradient flow to generators)
- The image pool provides historical fakes to the discriminators

In [ ]:
# Full training step demo
model = CycleGANModel(
    in_channels=3,
    out_channels=3,
    n_residual_blocks=3,      # small for demo
    n_discriminator_layers=3,
    lambda_cycle=10.0,
    lambda_identity=0.5,
)

# Optimizers: Adam with beta1=0.5 (standard for GANs)
optimizer_G = torch.optim.Adam(
    list(model.G_AB.parameters()) + list(model.G_BA.parameters()),
    lr=2e-4, betas=(0.5, 0.999),
)
optimizer_D = torch.optim.Adam(
    list(model.D_A.parameters()) + list(model.D_B.parameters()),
    lr=2e-4, betas=(0.5, 0.999),
)

# Image pools
pool_A = ImagePool(pool_size=50)
pool_B = ImagePool(pool_size=50)

# Simulate 5 training steps
model.train()
for step in range(5):
    real_A = torch.randn(2, 3, 128, 128)
    real_B = torch.randn(2, 3, 128, 128)
    
    losses = model.training_step(
        real_A, real_B,
        optimizer_G, optimizer_D,
        pool_A, pool_B,
    )
    
    print(f"Step {step}: "
          f"G_total={losses['loss_G']:.3f}  "
          f"D_A={losses['loss_D_A']:.3f}  "
          f"D_B={losses['loss_D_B']:.3f}  "
          f"cyc={losses['loss_cycle']:.3f}")

---
## 5. Learning Rate Scheduling

CycleGAN uses a **linear decay** schedule:
- For the first `n_epochs` epochs, the LR stays at the initial value.
- For the next `n_epochs_decay` epochs, the LR linearly decays to zero.

This is typically set as `n_epochs=100`, `n_epochs_decay=100` (total 200 epochs).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Simulate the linear decay LR schedule
initial_lr = 2e-4
n_epochs = 100
n_epochs_decay = 100
total_epochs = n_epochs + n_epochs_decay

lrs = []
for epoch in range(total_epochs):
    if epoch < n_epochs:
        lr = initial_lr
    else:
        lr = initial_lr * (1.0 - (epoch - n_epochs) / n_epochs_decay)
    lrs.append(lr)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(total_epochs), lrs, linewidth=2, color="#4CAF50")
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Learning Rate", fontsize=12)
ax.set_title("CycleGAN Linear Decay LR Schedule", fontsize=14)
ax.axvline(x=n_epochs, color="gray", linestyle="--", alpha=0.5, label=f"Decay starts (epoch {n_epochs})")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

---
## 6. CLI: `udm-epic3 train`

For full training, use the CLI command:

```bash
# Train baseline CycleGAN
udm-epic3 train --config configs/epic3_cyclegan.yaml
```

Key config options:

```yaml
training:
  n_epochs: 100           # constant LR phase
  n_epochs_decay: 100     # linear decay phase
  lr: 0.0002
  beta1: 0.5
  batch_size: 4
  lambda_cycle: 10.0
  lambda_identity: 0.5
  lambda_defect: 0.0      # disabled for baseline (see US 3.3)
  pool_size: 50
  save_interval: 10       # save checkpoint every 10 epochs
  sample_interval: 5      # save sample images every 5 epochs

model:
  n_residual_blocks: 9
  n_discriminator_layers: 3
```

Training outputs:
- Checkpoints in `outputs/epic3/checkpoints/`
- Sample translations in `outputs/epic3/samples/`
- TensorBoard logs in `outputs/epic3/logs/`

---
## Next steps

| Notebook | Topic |
|----------|------|
| `epic3_03_defect_aware.ipynb` | Add defect preservation loss for better defect translation |
| `epic3_04_evaluation.ipynb` | Quantitative evaluation with SSIM and Dice |